# Unit 3 group project: TrashNet + DINOv3 ConvNeXt Tiny

Compare frozen native-resolution global pooled features with a linear softmax
classifier, an 8-unit GELU ANN, and a training-majority baseline. Test images are
held out for the group's eventual single final model. All modeling uses TensorFlow/Keras.


## 1. Compatible dependencies and configuration

Keep TensorFlow 2.20, required by Kaggle's existing TensorFlow add-ons, while
using Keras 3.15.1 to read the saved artifact. Pin Protobuf 5.29.5 and Hugging
Face Hub 1.31.0 within all reported consumers' supported ranges. Install the
missing BigQuery Storage dependency and compatible Gym version. Run `pip check`
before imports: dependency conflicts fail setup instead of being ignored.
Remove unused Dopamine (whose Gym requirement conflicts with Gymnax) and Google
Colab (whose pandas/Jupyter requirements conflict with Kaggle's server stack).
Remove unused legacy MoviePy, whose Decorator requirement conflicts with MNE.


In [ ]:
# Install compatible pins before importing TensorFlow or dataset libraries.
import subprocess
import sys

# These unused preinstalled distributions impose mutually incompatible runtime
# constraints; removing them avoids changing Kaggle's pandas/Jupyter server.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y',
                'dopamine-rl', 'google-colab', 'moviepy'], check=True)
requirements = [
    'tensorflow==2.20.0', 'keras==3.15.1', 'datasets==5.1.0',
    'kagglehub==1.0.2', 'Pillow==12.3.0', 'protobuf==5.29.5',
    'huggingface-hub==1.31.0', 'google-cloud-bigquery-storage==2.35.0',
    'gym==0.26.2', 'decorator==5.3.1',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                '--disable-pip-version-check', *requirements], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'check'], check=True)


In [ ]:
import os
from pathlib import Path

# Set the backend before importing Keras, and reduce TensorFlow startup logging.
os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import json
import numpy as np
import tensorflow as tf
# Configure allocation before loading weights or creating any GPU tensors.
for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)
import keras
import kagglehub
from datasets import ClassLabel, Image, load_dataset
from IPython.display import display

DATASET_ID = "garythung/trashnet"
DATASET_REVISION = "1d4566ad6b22d2b5ef4eeba8f351b825d163eb11"
MODEL_HANDLE = "mataeoanderson/dinov3-convnext/keras/tiny/1"
MODEL_FILENAME = "dinov3_convnext_tiny.keras"
SAMPLE_INDEX = 0

# Relative paths work in Kaggle's writable working directory and local Jupyter.
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
SEED = 42
BATCH_SIZE = 64
MAX_EPOCHS = 50
PATIENCE = 5
keras.utils.set_random_seed(SEED)


## 2. Load TrashNet

Load the full training split from Hugging Face's published Parquet conversion of the same dataset. All four shard filenames and the conversion commit were verified using the Hub's repository metadata. Using Parquet preserves the published label schema and avoids the image-folder loader's label-inference error on this repository.

`load_dataset("parquet", ..., split="train")` returns a `Dataset`. Its `image` feature decodes to a PIL image, and `label` is an integer described by `ClassLabel`. The class names come from that schema. No train/validation/test split is created yet.

API reference: [Hugging Face Parquet loading](https://huggingface.co/docs/datasets/en/loading#parquet).

In [ ]:
# Explicit published shards avoid guessing labels from the source folder layout.
# This commit pins Hugging Face's Parquet conversion, including its image/label schema.
parquet_base = f"https://huggingface.co/datasets/{DATASET_ID}/resolve/{DATASET_REVISION}"
parquet_files = [
    f"{parquet_base}/default/train/0000.parquet",
    f"{parquet_base}/default/train/0001.parquet",
    f"{parquet_base}/default/train/0002.parquet",
    f"{parquet_base}/default/train/0003.parquet",
]
dataset = load_dataset("parquet", data_files={"train": parquet_files}, split="train")
assert isinstance(dataset.features["image"], Image)
label_feature = dataset.features["label"]
assert isinstance(label_feature, ClassLabel)
class_names = label_feature.names

print(dataset)
print("Classes:", class_names)
print("Number of images:", len(dataset))

## 3. Show one image and its label

Read the label through `ClassLabel.int2str`; the display is the original RGB image, before model normalization.

In [ ]:
# A row contains image (PIL) and label (integer); any other columns are unused.
sample = dataset[SAMPLE_INDEX]
image = sample["image"].convert("RGB")
label_id = int(sample["label"])
label_name = label_feature.int2str(label_id)

print(f"Sample {SAMPLE_INDEX}: {label_name} (label {label_id})")
print("Image size:", image.size)
display(image)
# Save the same image so a completed Kaggle run has a downloadable preview.
image.save(OUTPUT_DIR / "sample_image.png")

## 4. Load only the Tiny CNN

`kagglehub.model_download` returns the local directory for the exact version. On Kaggle it resolves the attached model input; locally it downloads into KaggleHub's cache. The `.keras` file contains the architecture and weights and requires no custom layers.

The backbone produces features, not TrashNet class predictions. Freeze it as a starting point for the group's future classification head.

API references: [KaggleHub model download](https://github.com/Kaggle/kagglehub#download-models), [Keras model loading](https://keras.io/api/models/model_saving_apis/model_saving_and_loading/).

In [ ]:
# Use the documented handle and filename; do not search for or load other variants.
model_dir = Path(kagglehub.model_download(MODEL_HANDLE))
model_file = model_dir / MODEL_FILENAME
if not model_file.is_file():
    raise FileNotFoundError(f"Expected model artifact: {MODEL_FILENAME}")

backbone = tf.keras.models.load_model(model_file, compile=False)
backbone.trainable = False
print("Loaded:", MODEL_HANDLE)
print("Model file:", model_file.name)
print("Parameters:", f"{backbone.count_params():,}")

## 5. Reusable experiment helpers

These helpers are maintained in `baselines.py` and embedded here by
`build_notebook.py` so this notebook runs by itself on Kaggle. `build_head` accepts
any fixed feature width; a spatial-feature experiment can reuse the head after
choosing its spatial representation/aggregation. No resize, crop, or padding is
applied. RGB scaling and ImageNet mean/std normalization remain.

API references: [Keras callbacks](https://keras.io/api/callbacks/),
[batch learning-rate schedules](https://keras.io/api/optimizers/learning_rate_schedules/learning_rate_schedule/),
[ROC/PR AUC](https://keras.io/api/metrics/classification_metrics/),
[Dataset stratified splitting](https://huggingface.co/docs/datasets/en/package_reference/main_classes#datasets.Dataset.train_test_split).


In [ ]:
"""Reusable TensorFlow heads, frozen-feature caching, and train/dev reporting.

The notebook embeds this module so Kaggle needs only Notebook.ipynb. No helper
accepts a test feature array: test evaluation belongs to a later final-model run.
"""
import json
import math
from pathlib import Path

import keras
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from datasets import DatasetDict


def split_dataset(published, seed=42, test_fraction=0.15, validation_fraction=0.15):
    """Honor DatasetDict author splits; stratify only missing partitions.

    Dataset.train_test_split returns DatasetDict with 'train' and 'test' keys.
    Its temporary 'test' partition becomes validation when splitting train/dev.
    Existing columns are preserved; row_id records original split membership.
    """
    splits = DatasetDict({name: data.add_column('row_id', [f'{name}:{i}' for i in range(len(data))])
                          for name, data in published.items()})
    if 'train' not in splits:
        raise ValueError('A published train split is required.')
    if 'validation' not in splits and 'dev' in splits:
        splits['validation'] = splits.pop('dev')
    if 'test' not in splits:
        partition = splits['train'].train_test_split(
            test_size=test_fraction, seed=seed, stratify_by_column='label')
        splits['train'], splits['test'] = partition['train'], partition['test']
        dev_fraction = validation_fraction / (1 - test_fraction)
    else:
        dev_fraction = validation_fraction
    if 'validation' not in splits:
        partition = splits['train'].train_test_split(
            test_size=dev_fraction, seed=seed, stratify_by_column='label')
        splits['train'], splits['validation'] = partition['train'], partition['test']
    ids = {name: set(splits[name]['row_id']) for name in ('train', 'validation', 'test')}
    assert not (ids['train'] & ids['validation'] or ids['train'] & ids['test'] or ids['validation'] & ids['test'])
    return splits


def preprocess_image(rgb_image):
    """Preserve every native RGB pixel; apply only scale and ImageNet normalization."""
    pixels = tf.convert_to_tensor(np.asarray(rgb_image.convert('RGB')), dtype=tf.float32) / 255.0
    mean = tf.constant([0.485, 0.456, 0.406], dtype=tf.float32)
    std = tf.constant([0.229, 0.224, 0.225], dtype=tf.float32)
    return ((pixels - mean) / std)[None, ...]


def prepare_pooled_backbone(backbone, mixed_precision=False):
    """Build a frozen pooled-only graph, preserving every pretrained weight.

    Eager Functional execution retains intermediate tensors until the call ends.
    At native multi-megapixel resolutions the expanded ConvNeXt activations can
    exhaust a T4 even with batch size one. Graph execution enables tensor-lifetime
    planning; pooled-only output also prunes the unused spatial-output branch.
    On GPU, mixed-float16 convolution/dense activations further halve their size.
    Normalization and global pooling stay float32; all variables stay float32.
    Heads and the shared RAM cache remain float32. No spatial pixels are changed.
    """
    if not isinstance(backbone.output, dict) or set(backbone.output) != {'pooled', 'feature_map'}:
        raise ValueError('Expected the Tiny backbone pooled/feature_map output contract.')
    pooled_model = keras.Model(backbone.inputs, backbone.output['pooled'], name='frozen_pooled_backbone')
    if mixed_precision:
        def clone_layer(layer):
            # Documented clone_function receives each non-Input layer instance.
            # Its config is copied explicitly; no checkpoint weights are modified.
            config = layer.get_config()
            config['dtype'] = 'float32' if isinstance(layer, (keras.layers.LayerNormalization, keras.layers.GlobalAveragePooling2D)) else 'mixed_float16'
            return layer.__class__.from_config(config)
        mixed_model = keras.models.clone_model(pooled_model, clone_function=clone_layer)
        mixed_model.set_weights(pooled_model.get_weights())
        pooled_model = mixed_model
    pooled_model.trainable = False
    assert not pooled_model.trainable_weights
    return pooled_model


def pooled_inference_function(pooled_model):
    """One dynamic-shape graph for all native image sizes, returning float32."""
    @tf.function(input_signature=[tf.TensorSpec((1, None, None, 3), tf.float32)])
    def infer(pixels):
        return tf.cast(pooled_model(pixels, training=False), tf.float32)
    return infer


def cache_pooled_features(backbone, splits):
    """One extraction pass, one inference per train/dev image, shared by all heads.

    The prepared frozen model returns only pooled (N,768), pruning spatial output.
    Single-image inference supports mixed native sizes without padding or resize.
    Dataset rows supply image, label, row_id; other columns are unused.
    """
    backbone.trainable = False
    assert not backbone.trainable_weights
    infer = pooled_inference_function(backbone)
    cache, shapes, calls = {}, {}, 0
    for name in ('train', 'validation'):
        data = splits[name]
        vectors = np.empty((len(data), 768), dtype=np.float32)
        labels = np.asarray(data['label'], dtype=np.int32)
        for i, row in enumerate(data):
            pixels = preprocess_image(row['image'])
            pooled = infer(pixels)
            assert tuple(pooled.shape) == (1, 768)
            tf.debugging.assert_all_finite(pooled, 'Non-finite pooled features')
            vectors[i] = pooled.numpy()[0]
            shape = str(tuple(pixels.shape[1:3]))
            shapes[shape] = shapes.get(shape, 0) + 1
            calls += 1
            if i == 0 or (i + 1) % 25 == 0 or i + 1 == len(data):
                print(f'{name}: cached {i + 1}/{len(data)}', flush=True)
        cache[name] = (vectors, labels)
    assert calls == len(splits['train']) + len(splits['validation'])
    print('Backbone inference calls:', calls, '| Native resolutions:', shapes)
    print('Feature cache MiB:', sum(x.nbytes + y.nbytes for x, y in cache.values()) / 2**20)
    return cache, {'inference_calls': calls, 'native_shapes': shapes, 'test_inference_calls': 0}


@keras.saving.register_keras_serializable(package='Unit3')
class WarmStableDecay(keras.optimizers.schedules.LearningRateSchedule):
    """Batch-based linear 10% warmup / 70% stable / 20% decay over 50 epochs.

    Early stopping may finish before decay; the schedule uses the planned budget,
    never the unknowable eventual stopping epoch. End learning rate is 1% of peak.
    """
    def __init__(self, total_steps, peak_lr=0.001):
        self.total_steps = int(total_steps)
        self.peak_lr = float(peak_lr)
        self.warmup = max(1, int(round(total_steps * 0.1)))
        self.decay_start = int(round(total_steps * 0.8))

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warm = self.peak_lr * (step + 1) / self.warmup
        progress = tf.clip_by_value((step - self.decay_start) / max(1, self.total_steps - self.decay_start - 1), 0., 1.)
        decay = self.peak_lr * (1. - 0.99 * progress)
        return tf.where(step < self.warmup, warm, tf.where(step < self.decay_start, self.peak_lr, decay))

    def get_config(self):
        return {'total_steps': self.total_steps, 'peak_lr': self.peak_lr}


def build_head(feature_dim, num_classes, hidden_units=(), name='linear'):
    """Dense softmax classifier for any fixed-width feature representation.

    A spatial experiment can supply vectors of another width or add its own
    spatial aggregation before this head. The default is multinomial logistic
    regression; hidden_units=(8,) adds one small GELU layer.
    """
    inputs = keras.Input(shape=(feature_dim,), name='features')
    x = inputs
    for i, units in enumerate(hidden_units):
        x = keras.layers.Dense(units, activation='gelu', name=f'hidden_{i}')(x)
    probabilities = keras.layers.Dense(num_classes, activation='softmax', name='probabilities')(x)
    return keras.Model(inputs, probabilities, name=name)


def feature_dataset(x, y, batch_size, training=False, seed=42):
    """Batch RAM features with a bounded TF thread pool for portable execution."""
    data = tf.data.Dataset.from_tensor_slices((x, y))
    if training:
        data = data.shuffle(len(y), seed=seed, reshuffle_each_iteration=True)
    options = tf.data.Options()
    options.threading.private_threadpool_size = 2
    return data.batch(batch_size).with_options(options).prefetch(1)


def train_head(spec, cache, num_classes, output_dir, batch_size=64, epochs=50, patience=5, seed=42):
    """Train only cached vectors, save strict best val_loss, reload best checkpoint.

    History.history contains every completed epoch including patience epochs.
    CSVLogger persists those rows even when EarlyStopping restores best weights.
    """
    keras.utils.set_random_seed(seed)
    x, y = cache['train']
    dev_x, dev_y = cache['validation']
    model = build_head(x.shape[1], num_classes, spec['hidden_units'], spec['name'])
    schedule = WarmStableDecay(epochs * math.ceil(len(y) / batch_size), spec['learning_rate']) if spec['wsd'] else spec['learning_rate']
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=schedule),
                  loss=keras.losses.SparseCategoricalCrossentropy(), metrics=['accuracy'])
    checkpoint = Path(output_dir) / f"{spec['name']}_best.keras"
    history = model.fit(feature_dataset(x, y, batch_size, True, seed),
        validation_data=feature_dataset(dev_x, dev_y, batch_size), epochs=epochs,
        callbacks=[keras.callbacks.ModelCheckpoint(checkpoint, monitor='val_loss', mode='min', save_best_only=True),
                   keras.callbacks.EarlyStopping(monitor='val_loss', mode='min', min_delta=0., patience=patience, restore_best_weights=True),
                   keras.callbacks.CSVLogger(Path(output_dir) / f"{spec['name']}_epochs.csv")], verbose=2)
    # Reloading verifies the on-disk deliverable and makes final scoring explicit.
    best = keras.models.load_model(checkpoint, compile=False)
    records = {k: [float(v) for v in values] for k, values in history.history.items()}
    metadata = {'name': spec['name'], 'parameters': best.count_params(), 'epochs_run': len(records['loss']),
                'best_epoch': int(np.argmin(records['val_loss'])) + 1,
                'stopped_early': len(records['loss']) < epochs, 'history': records, 'spec': spec}
    (Path(output_dir) / f"{spec['name']}_history.json").write_text(json.dumps(metadata, indent=2))
    return best, metadata


def evaluate_probabilities(labels, probabilities, class_names):
    """Argmax classification metrics plus one-vs-rest ROC/PR for each class.

    Keras AUC's documented TP/FP/TN/FN state yields curves at 2000 thresholds.
    Macro precision/recall/F1 gives each class equal weight; zero denominators
    produce 0. Cross entropy uses Keras epsilon clipping, including the one-hot
    majority baseline, whose wrong-class ideal loss would otherwise be infinite.
    """
    labels = np.asarray(labels, dtype=np.int32)
    probabilities = np.asarray(probabilities, dtype=np.float32)
    assert probabilities.shape == (len(labels), len(class_names))
    assert np.isfinite(probabilities).all()
    np.testing.assert_allclose(probabilities.sum(axis=1), 1., atol=1e-5)
    predictions = probabilities.argmax(axis=1)
    cm = tf.math.confusion_matrix(labels, predictions, num_classes=len(class_names)).numpy()
    tp = np.diag(cm)
    precision = np.divide(tp, cm.sum(axis=0), out=np.zeros(len(tp)), where=cm.sum(axis=0) != 0)
    recall = np.divide(tp, cm.sum(axis=1), out=np.zeros(len(tp)), where=cm.sum(axis=1) != 0)
    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros(len(tp)), where=precision + recall != 0)
    per_class, curves = [], {}
    for i, name in enumerate(class_names):
        truth = (labels == i).astype(np.int32)
        if truth.min() == truth.max():
            raise ValueError(f'{name} has no positive or no negative examples; AUC is undefined.')
        roc = keras.metrics.AUC(curve='ROC', num_thresholds=2000)
        pr = keras.metrics.AUC(curve='PR', num_thresholds=2000)
        roc.update_state(truth, probabilities[:, i])
        pr.update_state(truth, probabilities[:, i])
        # These four public metric variables are counts per threshold, not guesses.
        t, f, n, missed = [v.numpy() for v in (roc.true_positives, roc.false_positives, roc.true_negatives, roc.false_negatives)]
        curves[name] = {'fpr': np.divide(f, f+n, out=np.zeros_like(f), where=f+n != 0).tolist(),
                        'tpr': np.divide(t, t+missed, out=np.zeros_like(t), where=t+missed != 0).tolist(),
                        'precision': np.divide(t, t+f, out=np.ones_like(t), where=t+f != 0).tolist()}
        per_class.append({'class': name, 'support': int(cm[i].sum()), 'precision': float(precision[i]),
                          'recall': float(recall[i]), 'f1': float(f1[i]),
                          'roc_auc': float(roc.result()), 'pr_auc': float(pr.result())})
    return {'accuracy': float(np.mean(predictions == labels)),
            'loss': float(keras.losses.SparseCategoricalCrossentropy()(labels, probabilities)),
            'precision_macro': float(precision.mean()), 'recall_macro': float(recall.mean()), 'f1_macro': float(f1.mean()),
            'pr_auc_macro': float(np.mean([r['pr_auc'] for r in per_class])),
            'roc_auc_macro': float(np.mean([r['roc_auc'] for r in per_class])),
            'confusion_matrix': cm.tolist(), 'per_class': per_class, 'curves': curves}


def plot_reports(results, histories, class_names, output_dir):
    """Plot every registered run; adding future runs needs no plotting rewrite."""
    for name, history in histories.items():
        fig, ax = plt.subplots(figsize=(7, 4))
        epochs = np.arange(1, history['epochs_run'] + 1)
        ax.plot(epochs, history['history']['loss'], label='Train')
        ax.plot(epochs, history['history']['val_loss'], label='Validation')
        ax.axvline(history['best_epoch'], color='grey', linestyle='--', label='Best checkpoint')
        ax.set(title=f'{name}: all completed epochs', xlabel='Epoch', ylabel='Cross entropy')
        ax.legend(); fig.tight_layout(); fig.savefig(Path(output_dir) / f'{name}_learning_curve.png'); plt.show()
    for split in ('train', 'validation'):
        for name, reports in results.items():
            report = reports[split]
            fig, axes = plt.subplots(1, 3, figsize=(18, 5))
            cm = np.asarray(report['confusion_matrix'])
            axes[0].imshow(cm, cmap='Blues')
            axes[0].set(xticks=range(len(class_names)), yticks=range(len(class_names)),
                        xticklabels=class_names, yticklabels=class_names, xlabel='Predicted', ylabel='True', title='Confusion matrix')
            axes[0].tick_params(axis='x', rotation=45)
            for i in range(len(class_names)):
                for j in range(len(class_names)):
                    axes[0].text(j, i, str(cm[i,j]), ha='center', va='center', color='white' if cm[i,j] > cm.max()/2 else 'black')
            for label, curve in report['curves'].items():
                axes[1].plot(curve['fpr'], curve['tpr'], label=label)
                axes[2].plot(curve['tpr'], curve['precision'], label=label)
            axes[1].plot([0, 1], [0, 1], 'k--')
            axes[1].set(xlabel='False positive rate', ylabel='Recall', title='One-vs-rest ROC')
            axes[2].set(xlabel='Recall', ylabel='Precision', title='One-vs-rest PR')
            axes[1].legend(fontsize=8); axes[2].legend(fontsize=8)
            fig.suptitle(f'{name} — {split}'); fig.tight_layout()
            fig.savefig(Path(output_dir) / f'{name}_{split}_metrics.png'); plt.show()


## 6. Stratified splits and class counts

The pinned Parquet conversion contains only `train`. Create 70% train / 15%
validation / 15% test with seed 42 and stratified labels. The reusable splitter
also accepts author-provided train/test or train/validation/test partitions and
preserves them, splitting train only when a partition is missing. Save row IDs
so future experiments use exactly the same partitions. Only counts/IDs are
inspected for test; its images never reach the backbone or classifiers.


In [ ]:
import pandas as pd

splits = split_dataset(DatasetDict({'train': dataset}), seed=SEED)
counts = pd.DataFrame({name: np.bincount(np.asarray(data['label']), minlength=len(class_names))
                       for name, data in splits.items()}, index=class_names)
display(counts)
print('Split sizes:', {name: len(data) for name, data in splits.items()})
(OUTPUT_DIR / 'split_manifest.json').write_text(json.dumps(
    {'seed': SEED, 'dataset_revision': DATASET_REVISION,
     'row_ids': {name: list(data['row_id']) for name, data in splits.items()}}, indent=2))


## 7. Extract native-resolution pooled features once

One forward pass per train/validation image, performed before either head trains.
Both heads share the same float32 RAM cache. Spatial maps are discarded. The
backbone is fully frozen and called with `training=False`. No test feature cache
is created, and the former extra sample forward pass has been removed.

Use a pooled-only TensorFlow graph to avoid retaining eager intermediate maps.
GPU convolution/dense activations use mixed float16; pretrained variables,
normalization, global pooling, cached vectors, and classifier heads stay float32.
This reduces memory without resizing or changing any pretrained weight.


In [ ]:
import gc

use_mixed_precision = bool(tf.config.list_physical_devices('GPU'))
backbone = prepare_pooled_backbone(backbone, mixed_precision=use_mixed_precision)
gc.collect()
print('Backbone execution: pooled-only graph | Mixed activations:', use_mixed_precision)
print('Backbone input:', backbone.input_shape, '| Output:', backbone.output_shape)
feature_cache, extraction_report = cache_pooled_features(backbone, splits)
extraction_report['mixed_precision_activations'] = use_mixed_precision
(OUTPUT_DIR / 'feature_extraction.json').write_text(json.dumps(extraction_report, indent=2))


## 8. Train linear softmax, then lightweight ANN

Both use batch size 64, at most 50 epochs, and strict validation-loss improvement
with patience 5. Best checkpoints are saved at each improving epoch and reloaded
for final scoring. All epoch losses, including patience epochs, remain available.

The linear head is multinomial logistic regression (768 → 6 softmax). The ANN is
768 → 8 GELU → 6 softmax: 6,206 parameters versus 4,614 for the linear head. Adam
uses 0.001 for both; the ANN uses linear WSD across planned optimizer updates:
10% warmup, 70% stable, 20% decay to 1% of peak. Early stopping can end before decay.
Add future configurations to `RUN_SPECS` to reuse training, evaluation, and plots.


In [ ]:
RUN_SPECS = [
    {'name': 'logistic_regression', 'hidden_units': (), 'learning_rate': 0.001, 'wsd': False},
    {'name': 'tiny_ann', 'hidden_units': (8,), 'learning_rate': 0.001, 'wsd': True},
]
trained_heads, histories = {}, {}
for spec in RUN_SPECS:
    print('Training:', spec['name'], flush=True)
    trained_heads[spec['name']], histories[spec['name']] = train_head(
        spec, feature_cache, len(class_names), OUTPUT_DIR,
        batch_size=BATCH_SIZE, epochs=MAX_EPOCHS, patience=PATIENCE, seed=SEED)
    print({key: value for key, value in histories[spec['name']].items() if key != 'history'})


## 9. Final train/validation metrics and comparison

Final scores use reloaded best-validation checkpoints. Predictions are softmax
argmax. Precision, recall, F1, ROC-AUC, and PR-AUC are macro averages; per-class
scores and confusion matrices are also retained. AUC uses Keras interpolation at
2,000 thresholds for each one-vs-rest class. These are numerical approximations.

The majority class is selected from training labels only and always receives
probability 1. Keras clips zero probabilities for finite cross-entropy reporting;
ideal wrong-class loss for this hard baseline would be infinite. No test scoring
occurs. Training metrics are diagnostics; validation is the comparison set.


In [ ]:
majority_class = int(np.bincount(feature_cache['train'][1], minlength=len(class_names)).argmax())
print('Training majority class:', class_names[majority_class])
results = {'majority_class': {}}
for split_name, (x, y) in feature_cache.items():
    probabilities = np.zeros((len(y), len(class_names)), dtype=np.float32)
    probabilities[:, majority_class] = 1.
    results['majority_class'][split_name] = evaluate_probabilities(y, probabilities, class_names)
for name, head in trained_heads.items():
    results[name] = {}
    for split_name, (x, y) in feature_cache.items():
        probabilities = head.predict(feature_dataset(x, y, BATCH_SIZE), verbose=0)
        results[name][split_name] = evaluate_probabilities(y, probabilities, class_names)

metric_names = ['accuracy', 'f1_macro', 'pr_auc_macro', 'roc_auc_macro', 'recall_macro', 'precision_macro', 'loss']
comparison = pd.DataFrame([{'model': name, 'split': split_name,
                           **{key: report[key] for key in metric_names}}
                          for name, reports in results.items() for split_name, report in reports.items()])
display(comparison.set_index(['split', 'model']).sort_index())
comparison.to_csv(OUTPUT_DIR / 'comparison.csv', index=False)
(OUTPUT_DIR / 'metrics.json').write_text(json.dumps(results, indent=2))
for name, reports in results.items():
    print(name, '— validation per-class metrics')
    display(pd.DataFrame(reports['validation']['per_class']).set_index('class'))


## 10. Learning curves, confusion matrices, ROC and PR curves

Every run gets the same plots. Learning curves retain all completed epochs and
mark the best checkpoint. Overfitting may appear as training loss falling while
validation loss rises; this behavior is not guaranteed in five patience epochs.


In [ ]:
plot_reports(results, histories, class_names, OUTPUT_DIR)
print('Saved artifacts:', [path.name for path in sorted(OUTPUT_DIR.iterdir())])
print('Finished. Test inference and scoring calls: 0.')
